# Consent HIU Notify resilience — M3-26

Tests `server/callbacks/services/consent_hiu_notify_service.py`'s `process_consent_hiu_notify()`, the
handler for a multi-hospital consent grant notification (one `consentArtefacts` entry per hospital/HIP
covered by the grant).

Uses the shared `harness.py` (storage isolation via `activate_scratch_storage()`, `FakeResponse` for stubbed
outbound calls) established in `set_a_idempotency_M2-9_M2-10_M2-11_M2-12_M2-13.ipynb` and reused across every
notebook in this suite.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "server").exists():
    REPO_ROOT = Path("__file__").resolve().parents[2] if Path("__file__").exists() else Path.cwd().parents[2]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("repo root on sys.path:", REPO_ROOT)
assert (REPO_ROOT / "server" / "callbacks").exists(), (
    "Couldn't find server/callbacks/ from here -- open this notebook with the repo root as the "
    "Jupyter working directory, or edit REPO_ROOT above by hand."
)

import asyncio
from unittest.mock import patch

import harness

import server.callbacks.services.consent_hiu_notify_service as consent_notify_service
from server.callbacks.repository.pending_consent_request_repository import (
    save_pending_consent_request, link_consent_request_id,
)


repo root on sys.path: C:\Users\hp\Desktop\Aayush\repo


---
## M3-26 — A network hiccup fetching ONE hospital's consent must not silently drop the WHOLE grant

**Real-world scenario:** a patient grants consent spanning 3 hospitals in one notification. ABDM sends us
one `consentArtefacts` entry per hospital, and we call `fetch_consent()` once per entry, in a loop. Before
this fix, that loop had no per-entry error handling -- if hospital 2's `fetch_consent()` call hit a
transient network failure, the exception propagated all the way out to this function's OUTER
try/except, which just logs and returns WITHOUT ever reaching the `send_consent_hiu_on_notify()` ack. That
doesn't just fail hospital 2: it silently drops the ack for ALL THREE hospitals, including hospital 1 whose
fetch had already succeeded a moment earlier (that side effect already happened and can't be undone, but
ABDM is never told) and hospital 3 which never even got a chance to run. From ABDM's side, the entire
multi-hospital grant looks like it was never received.

**Fix:** each artefact's own `fetch_consent()` call is now wrapped in its own try/except -- one artefact's
failure is logged and skipped, every other artefact in the same notification still gets its fetch attempted,
and the ack step always runs afterward for whichever artefacts have an id, regardless of how any individual
fetch went.

**Pass criteria:** with hospital 2's fetch rigged to raise a `ConnectionError`, all 3 artefacts' fetches are
still attempted (hospital 2's failure doesn't stop 1 and 3), and ABDM still receives one ack covering all 3
artefacts.

In [2]:
harness.activate_scratch_storage("m3_26")

save_pending_consent_request("orig-req-m3-26", {"hiu_id": "HIU-1"})
link_consent_request_id("orig-req-m3-26", "consent-req-m3-26")

fetch_calls = []
def flaky_fetch_consent(hiu_id, consent_id):
    fetch_calls.append(consent_id)
    if consent_id == "consent-hospital-2":
        raise ConnectionError("simulated network hiccup talking to ABDM for hospital 2's fetch")
    return harness.FakeResponse(202)

ack_calls = []
def fake_send_consent_hiu_on_notify(acknowledgements, request_id):
    ack_calls.append(acknowledgements)
    return harness.FakeResponse(202)

callback_data = {
    "headers": {"request-id": "req-m3-26"},
    "body": {"notification": {
        "consentRequestId": "consent-req-m3-26",
        "status": "GRANTED",
        "consentArtefacts": [
            {"id": "consent-hospital-1"},
            {"id": "consent-hospital-2"},  # this one's fetch will raise
            {"id": "consent-hospital-3"},
        ],
    }},
}

with patch.object(consent_notify_service, "fetch_consent", flaky_fetch_consent), \
     patch.object(consent_notify_service, "send_consent_hiu_on_notify", fake_send_consent_hiu_on_notify):
    await consent_notify_service.process_consent_hiu_notify(callback_data)

harness.check("all 3 artefacts' fetch_consent() were attempted (hospital 2's failure didn't stop the others)",
              fetch_calls == ["consent-hospital-1", "consent-hospital-2", "consent-hospital-3"])
harness.check("ABDM still got an ack for ALL 3 artefacts, including the ones whose fetch succeeded around the failure",
              len(ack_calls) == 1 and len(ack_calls[0]) == 3)


2026-08-14 21:22:17  -> Consent status notification received from ABDM (POST /api/v3/hiu/consent/request/notify)
2026-08-14 21:22:17  -> Extracted consentRequestId and status (GRANTED) -- 3 artefact(s)
2026-08-14 21:22:17     [API] Fetching granted consent artefact consent-hospital-1 -- POST .../consent/v3/fetch -> 202
2026-08-14 21:22:17     [ERROR] fetch_consent() failed for artefact consent-hospital-2 (consentRequestId consent-req-m3-26): simulated network hiccup talking to ABDM for hospital 2's fetch -- continuing with the remaining artefact(s) in this notification.
2026-08-14 21:22:17     [API] Fetching granted consent artefact consent-hospital-3 -- POST .../consent/v3/fetch -> 202
2026-08-14 21:22:17  -> Consent granted -- fetch attempted for each artefact
2026-08-14 21:22:17     [API] Acknowledging Consent Notification to ABDM -- POST .../hiu/on-notify -> 202
2026-08-14 21:22:17     [WAITING] Waiting for ABDM's on-fetch callback with full consent detail


[harness] scratch storage active at: C:\Users\hp\AppData\Local\Temp\edge_case_scratch_m3_26_w_hp99ue
[harness] (NOT the repo's real storage/ directory -- nothing here touches that)
PASS -- all 3 artefacts' fetch_consent() were attempted (hospital 2's failure didn't stop the others)
PASS -- ABDM still got an ack for ALL 3 artefacts, including the ones whose fetch succeeded around the failure


True

---
## Summary

M3-26 confirmed a real bug (silent whole-grant drop on any single artefact's network hiccup) and fixed it
with per-artefact error isolation. All checks above PASS against the real, fixed service code.